In [ ]:
"""\
# Copyright (C) 2024 Jesús Bautista Villar <jesbauti20@gmail.com>
"""

In [ ]:
# If executed from Google Colab
# !git clone https://github.com/jesusBV20/MRS-SS_souce_seeking.git
# !rsync -a MRS-SS_souce_seeking/ .
# !rm -r MRS-SS_souce_seeking

In [ ]:
# If you want to use latex with matplotlib
# !apt install -y texlive texlive-latex-extra texlive-fonts-recommended dvipng cm-super
# !pip install -y latex

# Libraries and global variables

In [2]:
# ----------------------------------------------------------------------
# Import main libraries
# ----------------------------------------------------------------------

import numpy as np
import sys
import os

# Graphic tools
import matplotlib.pyplot as plt
import matplotlib

# Animation tools
from IPython.display import HTML

# -------------------------------------------------------------------------------------
# Import from the Swarm Systems Lab Simulator

# Tell matplotlib to use latex
from ssl_simulator.visualization import set_paper_parameters
set_paper_parameters(fontsize=14)

# Main utility functions used along the notebook
from ssl_simulator import create_dir

# ----------------------------------------------------------------------
# Source Seeking Tools

module_path = os.path.abspath("..")
if module_path not in sys.path:
    sys.path.append(module_path)

from sourceseeking_2d.toolbox import *
from sourceseeking_2d.scalar_field import *

from sourceseeking_2d.simulations import *

from sourceseeking_2d.plot_article import *
from sourceseeking_2d.plot_simulations import *

# Fix the random seed
np.random.seed(2023)

# -------------------------------------------------------------------------------------
# Define constants for file paths
OUTPUT_DIR = os.path.join("..", "output")
create_dir(OUTPUT_DIR)

The directory '../output' already exists!


---
# The Ascending Direction

## Lemma: Ascending direction 

In [ ]:
# Parameters --
N = 10
l1, l2 = 10, 10

# Scalar field
mu0 = np.array([16,10]) 

# ----------------------------------------------------------------------
# Generating the scalar field
# ----------------------------------------------------------------------
max_int = 100
mu = mu0 + 6 * (np.random.rand() - 0.5)
dev = 40 + 20 * (np.random.rand() - 0.5)

psi = np.pi * np.random.rand()
a, b = 1 + 5*(np.random.rand()+1), 1 + 3*(np.random.rand()+1)

# Generating...
S = -np.array([[a,0],[0,b]])
R =  M_rot(psi)
R2 = M_rot(psi)

sigma_func = sigma_gauss(mu=mu, max_intensity=max_int, dev=dev, S=S, R=R)
sigma_test = sigma(sigma_func)
sigma_test.rot = R2

# ----------------------------------------------------------------------
# Generate the formation
# ----------------------------------------------------------------------
phi = 2 * np.pi * np.random.rand(N)
p0 = mu0 + 40 * (np.random.rand(2) - 0.5)
px = p0[0] + l1 * (np.random.rand(N) - 0.5)
py = p0[1] + l2 * (np.random.rand(N) - 0.5)

P = np.array([px,py]).T
pc = np.sum(P, axis=0)/N

sigma_values = sigma_test.value(P)

# Compute L_sigma
l_sigma = L_sigma(P - pc, sigma_values)
l_sigma = l_sigma/np.sqrt(l_sigma[0]**2 + l_sigma[1]**2)

# Compute L_sigma^1
l1_vec = sigma_test.draw_L1(pc, P)
l1_vec = l1_vec/np.sqrt(l1_vec[0]**2 + l1_vec[1]**2)

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
# Generate the plot
fig = plt.figure(figsize=(16, 8), dpi=80)
ax = fig.subplots()

# Draw the scalar field
sigma_test.draw(fig=fig, ax=ax, xlim=60, ylim=40, n=300, contour_levels=20)

# Axis configuration
ax.set_xlim([-30,70])
ax.set_ylim([-20,30])
ax.set_xlabel(r"$P_x$ [L]")
ax.set_ylabel(r"$P_y$ [L]")
ax.grid(True)

# Draw the agents
for n in range(N):
    icon = unicycle_patch([px[n], py[n]], phi[n], "royalblue", **KW_PATCH)
    ax.add_patch(icon)

# Plot S region
plot_sregion(ax, pc, l1, l2)

# Draw the gradient at pc, L^1 and L
sigma_test.draw_grad(pc, ax, width=0.003, scale=15)
vector2d(ax, pc, l_sigma*6, c="red"  , **KW_ARROW)
vector2d(ax, pc, l1_vec *6, c="green", **KW_ARROW)

# Generate the legend
arr1 = plt.scatter([],[],c='k'  ,marker=r'$\uparrow$',s=60)
arr2 = plt.scatter([],[],c='red',marker=r'$\uparrow$',s=60)
arr3 = plt.scatter([],[],c='green',marker=r'$\uparrow$',s=60)

leg = Legend(ax, [arr1, arr2, arr3], 
             [r"$\nabla \sigma (p_c)$ (Non-computed)",
              r"$L_{\sigma}$: Actual computed ascending direction",
              r"$L_1$ (Non-computed)"],
            loc="upper left", prop={'size': 12})

ax.add_artist(leg)

# Labels with the sigma measured
#for i in range(N):
#    ax.text(P[i,0]-1.5, P[i,1]+0.7, "{0:.2f}".format(sigma_values[i]))

 # Show the plot!
plt.show()

## Collorary: Clusters

In [ ]:
# Parameters --
clusters = [
    cluster(np.array([
        [5, 6], [3, 4.8], [4, 4], [4, 7], [3.5, 6], [3.8, 5], [2.3, 5.8]
        ])),
    cluster(np.array([
        [6.5, 1], [7, 1.5], [7.5, 2.5], [6.2, 2.1], [7, 2.2], [6.6, 2.3]
        ])),
    cluster(np.array([
        [8, 6], [7.5, 7], [8.5, 8], [8.4, 6.2], [9, 6.6], [8.7, 7.3]
        ]))
]

# Center and radius of the decagon
center = np.array([-10, 0])
radius = 10

# Angles between each point
angles = np.linspace(0, 2*np.pi, 12, endpoint=False)

# ----------------------------------------------------------------------
# Generating the scalar field
# ----------------------------------------------------------------------

sigma_func = sigma_gauss(mu=[30,20], max_intensity=100, dev=20)
sigma_test = sigma(sigma_func)

# ----------------------------------------------------------------------
# Generate the formation
# ----------------------------------------------------------------------

# Calculate the x and y coordinates of each point
x_coords = center[0] + radius * np.cos(angles)
y_coords = center[1] + radius * np.sin(angles)

# Combine the x and y coordinates into a single array
deca = np.column_stack((x_coords, y_coords))

clusters_deca = [
    cluster(deca[6:9]),
    cluster(deca[0:3]),
    cluster(deca[9:12]),
    cluster(deca[3:6])
]

# ------------------------------
# Calling the plotting function
# ------------------------------
clusters_plot(clusters_deca, sigma_test)

---
# Sensibility and Observability

## Proposition: Degenerated line formations

In [ ]:
# Parameters --
p0 = np.array([-10,-3])

nx, ny = 10, 10
a1, a2 = 0.9, 0.8
lx, ly = 10, 10

# ----------------------------------------------------------------------
# Generating the scalar field
# ----------------------------------------------------------------------

sigma_func = sigma_gauss(mu=[30,20], max_intensity=100, dev=20)
sigma_test = sigma(sigma_func)

# ----------------------------------------------------------------------
# Generate the formation
# ----------------------------------------------------------------------
a_x1d = calculate_a(nx, a1)
a_y1d = calculate_a(ny, a2)

P_x = a_x1d.reshape(1, np.size(a_x1d))
P_x = np.vstack((a_x1d*lx, np.zeros((1, np.size(a_x1d)))))

P_y = a_y1d.reshape(1, np.size(a_y1d))
P_y = np.vstack((np.zeros((1, np.size(a_y1d))), a_y1d*ly))

P = np.hstack((P_x,P_y)).T + p0
pc = np.mean(P, axis=0)

# Compute the measured sigma values
sigma_values = sigma_test.value(P)

# Compute L_sigma
l_sigma = L_sigma(P - pc, sigma_values)
l_sigma = l_sigma/np.sqrt(l_sigma[0]**2 + l_sigma[1]**2)

# Compute L_sigma^1
l1_vec = sigma_test.draw_L1(pc, P)
l1_vec = l1_vec/np.sqrt(l1_vec[0]**2 + l1_vec[1]**2)

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------

# Generate the plot
fig = plt.figure(figsize=(16, 8), dpi=80)
ax = fig.subplots()

# Draw the scalar field
sigma_test.draw(fig=fig, ax=ax, xlim=60, ylim=40, n=300, contour_levels=20)

# Axis configuration
ax.set_xlim([-30,70])
ax.set_ylim([-20,30])
ax.set_xlabel(r"$P_x$ [L]")
ax.set_ylabel(r"$P_y$ [L]")
ax.grid(True)

title = r"$N_\parallel$ = {0:d}, $N_-$ = {1:d}, ".format(nx,ny)
title = title + r"$\quad a_1^\parallel$ = {0:.1f}, $a_1^-$ = {1:.1f}".format(lx,ly)
ax.set_title(title)

# Draw the agents
for n in range(nx + ny):
    ax.add_patch(plt.Circle(P[n], 0.3, color="royalblue", alpha=0.8))

# Draw the gradient at pc, L^1 and L
sigma_test.draw_grad(pc, ax, width=0.003, scale=15)
vector2d(ax, pc, l_sigma*6, c="red"  , **KW_ARROW)
vector2d(ax, pc, l1_vec *6, c="green", **KW_ARROW)

# Generate the legend
arr1 = plt.scatter([],[],c='k'  ,marker=r'$\uparrow$',s=60)
arr2 = plt.scatter([],[],c='red',marker=r'$\uparrow$',s=60)
arr3 = plt.scatter([],[],c='green',marker=r'$\uparrow$',s=60)

leg = Legend(ax, [arr1, arr2, arr3], 
             [r"$\nabla \sigma (p_c)$ (Non-computed)",
              r"$L_{\sigma}$: Actual computed ascending direction",
              r"$L_1$ (Non-computed)"],
            loc="upper left", prop={'size': 12})
ax.add_artist(leg)

# Show the plot!
plt.show()

## Lemma: Poly formations

In [ ]:
# Parameters --
r = 1

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
fig = plt.figure(figsize=(18, 10), dpi=100)
ax  = fig.subplots(2,3)

plot_polyreg(ax[0,0],3,r,legend=True)
plot_polyreg(ax[0,1],4,r)
plot_polyreg(ax[0,2],5,r)
plot_polyreg(ax[1,0],6,r,xlab=True,ylab=True)
plot_polyreg(ax[1,1],30,r)
plot_polyreg(ax[1,2],100,r)

# Show the plot!
plt.show()

## Proposition: Poly formations

In [ ]:
# Parameters --
N1, N2 = 3, 5
r = 20

rt_ang = [0, np.pi/3, np.pi/2]

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
fig = plt.figure(figsize=(18, 10), dpi=100)
ax  = fig.subplots(2,3)

plot_polyreg(ax[0,0],N1,r,rt_ang[0],title_full=True,legend=True)
plot_polyreg(ax[0,1],N1,r,rt_ang[1],title_full=True)
plot_polyreg(ax[0,2],N1,r,rt_ang[2],title_full=True)
plot_polyreg(ax[1,0],N2,r,rt_ang[0],title_full=True,xlab=True,ylab=True)
plot_polyreg(ax[1,1],N2,r,rt_ang[1],title_full=True)
plot_polyreg(ax[1,2],N2,r,rt_ang[2],title_full=True)

# Show the plot!
plt.show()

## Proposition: Variance

In [ ]:
# Parameters --
lx = [2, 4]
ly = [2, 2]

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
fig = plt.figure(figsize=(18, 10), dpi=100)
ax  = fig.subplots(1,2)

plot_rect(ax[0],lx[0],ly[0],legend=True,xlab=True,ylab=True)
plot_rect(ax[1],lx[1],ly[1])

# Show the plot!
plt.show()


## Proposition: Symmetries in the continuum

In [ ]:
# Parameters --
N = [1000, 500, 100]
r = [10,10,10]
b = [6,6,6]

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
fig = plt.figure(figsize=(18, 10), dpi=100)
ax  = fig.subplots(1,3)

plot_flower(fig, ax[0],N[0],r[0],b=b[0],legend=1,xlab=True,ylab=True)
plot_flower(fig, ax[1],N[1],r[1],b=b[1],legend=2)
plot_flower(fig, ax[2],N[2],r[2],b=b[2],legend=3)

# Show the plot!
plt.show()

### Batman

In [ ]:
# Parameters --
N = 1000
lims = [[10,10], [20,10], [10,35]]
lims_ax = [-20, 20, -20, 20]
# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
fig = plt.figure(figsize=(18, 10), dpi=100)
ax  = fig.subplots(1,3)

plot_batman(fig,ax[0],N,lims[0],legend=1,xlab=True,ylab=True,lims_ax=lims_ax)
plot_batman(fig,ax[1],N,lims[1],legend=2,lims_ax=lims_ax)
plot_batman(fig,ax[2],N,lims[2],legend=3,lims_ax=lims_ax)

# Show the plot!
plt.show()

---
# Distributed estimation of the centroid

## Plot

In [ ]:
# Parameters --
N = [3, 6, 9]
r = 1

tf = [1, 2] # s
f_inv = [1e6, 1e7] # 1e6 Hz = 1 MHz

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
fig = plt.figure(figsize=(18, 10), dpi=100)
ax  = fig.subplots(1,3)

plot_centroid(ax[0],N[0],r,tf[0],f_inv[0], legend=True,xlab=True,ylab=True)
plot_centroid(ax[1],N[1],r,tf[0],f_inv[0])
plot_centroid(ax[2],N[1],r,tf[1],f_inv[0])

# Show the plot!
plt.show()

## Animation

In [ ]:
# Parameters --
N = 9
r = 1

tf = 10 # s
f_inv = 1e5 # 1e6 Hz = 1 MHz

# ----------------------------------------------------------------------
# Plotting
# ----------------------------------------------------------------------
anim = anim_centroid(N,r,tf,f_inv)
HTML(anim.to_html5_video())

---
# Source Seeking Simulations

*A single cluster that modifies its shape and number of agents*

Mission characteristics:

* We have a single swarm.
* We will be able to generate N agents distributed in different shapes around a centroid.
* The swarm can modify its shape throughout the simulation.

## Simulation Class 1: Source-seeking with **single integrators**

### **SIM 1**: The geometry of the formation changes to avoid obstacles

In [ ]:
# ----------------------------------------------------------------------
# Generating the scalar field
# ----------------------------------------------------------------------
n = 2
max_int = 20
mu = [40,40]
dev = 10

sigma_func  = sigma_gauss(mu, max_intensity=max_int, dev=dev, n=n)
sigma_func = sigma_nonconvex(k=0.04, dev=dev, mu=mu)
sigma_field = sigma(sigma_func)

# ----------------------------------------------------------------------
# Simulation parameters
# ----------------------------------------------------------------------
dt = 0.1
t0 = 0
t_sim_final = 120

# Initialize the simulation
obstacles = [[0,0,10], [10,-35,5]]

# Number of agents and initial states of the agents
n_agents = 250
rc0 = [-35, -50]
r, h = 10, 2
lims = [15, 2]
border_noise = 0.6
d_max = 11 #!!

p0_cir = circular_distrib(n_agents, n, [0,0], r, h, border_noise)
p0_sqr = XY_distrib(n_agents, n, [0,0], lims, border_noise)
p0_bat = batman_distrib(n_agents, [0,0], [13,7])

p0 = rc0 + p0_cir
v0 = 1.5

In [ ]:
# ----------------------------------------------------------------------
# Numerical simulation
# ----------------------------------------------------------------------

# Initialize the simulation
sim = simulation_class1(sigma_field, n_agents, [t0, p0, v0], dt, True, obstacles, ang_noise=10, it_noise=2)

# Initialize the data collector
data_labels = ["pf", "rc", "e", "d", "sigma", "active", "l_sigma", 
               "rc_grad", "field_rot"]
data_col = data_collector(sim, data_labels)
data_col.collect()

# Execute numerical simulation
t1, t2 = 20, 45
while (sim.tf <= t_sim_final - dt/10):
  if (sim.tf >= t1) and (sim.tf <=t1+1):
    sim.Xd = p0_sqr
  if (sim.tf >= t2) and (sim.tf <=t2+1):
    sim.Xd = p0_cir

  # Integrate new step
  sim.int_step()
  data_col.collect()

  # Check if any agent has to die!!
  ddata = data_col.get("d")[-1,:]
  for i in range(sim.N):
    if ddata[i] is not None:
      if ddata[i] > d_max:
        sim.active[i] = False
      else:
        random_num = np.random.random()
        if random_num > 0.999:
          sim.active[i] = False

print(np.sum(sim.active))

In [ ]:
# ----------------------------------------------------------------------
# Plot few steps of the simulation
# ----------------------------------------------------------------------
plot_class1(data_col, sim, t_list=[10,45,65,90], d_max=d_max)

In [18]:
# ----------------------------------------------------------------------
# Animate the simulation
# ----------------------------------------------------------------------
anim = anim_class1(data_col, sim, anim_tf=110, res_label="2K", d_max=d_max)
HTML(anim.to_html5_video()) # It takes a looot of time...

### **SIM 2**: The swarm formation rotates

In [ ]:
# ----------------------------------------------------------------------
# Maneuverability parameters
# ----------------------------------------------------------------------
limx, limy = 10, 2
psi = 45 * np.pi / 180
ab_ = 4

# ----------------------------------------------------------------------
# Scalar field generation
# ----------------------------------------------------------------------
n = 2
max_int = 20
dev = 10
mu = [40,40]

S = -np.array([[1,0],[0,ab_]])
R = M_rot(psi)

# Define the scalar field
sigma_func = sigma_gauss(mu=mu, max_intensity=max_int, dev=dev, S=S, R=R)
sigma_field = sigma(sigma_func)

# ----------------------------------------------------------------------
# Simulation parameters
# ----------------------------------------------------------------------
dt = 0.1
t0 = 0
t_sim_final = 100
n_agents = 200

# Generate different distributions
rc0 = [-35, -50]
border_noise = 0.6
lims = [limx, limy]
R45 = M_rot(-45 * np.pi/180)
R90 = M_rot(-90 * np.pi/180)

p1 = XY_distrib(n_agents, n, [0,0], lims, border_noise)
p2 = Q_prod_xi(R45,p1)
p3 = Q_prod_xi(R90,p1)

# Initial state of the agents and number of agent
p0 = rc0 + p2
v0 = 2

In [ ]:
# ----------------------------------------------------------------------
# Numerical simulation
# ----------------------------------------------------------------------

# Initialize the simulation
sim = simulation_class1(sigma_field, n_agents, [t0, p0, v0], dt, True)

# Initialize the data collector
data_labels = ["pf", "rc", "e", "d", "sigma", "active", "l_sigma", 
               "rc_grad", "field_rot"]
data_col = data_collector(sim, data_labels)
data_col.collect()

# Execute numerical simulation
t1, t2 = 10, 30
while (sim.tf <= t_sim_final - dt/10):
  if (sim.tf >= t1) and (sim.tf <= t1+1):
    sim.Xd = p1
  if (sim.tf >= t2) and (sim.tf <= t2+1):
    sim.Xd = p3

  # Integrate new step
  sim.int_step()
  data_col.collect()

In [ ]:
# ----------------------------------------------------------------------
# Plot few steps of the simulation
# ----------------------------------------------------------------------
plot_class1(data_col, sim, t_list=[10,30,60])

In [ ]:
# ----------------------------------------------------------------------
# Animate the simulation
# ----------------------------------------------------------------------
anim = anim_class1(data_col, sim, anim_tf=100)
HTML(anim.to_html5_video()) # It takes a looot of time...

### **SIM 3**: The scalar field rotates to maneuver batman formation

In [ ]:
# ----------------------------------------------------------------------
# Maneuverability parameters
# ----------------------------------------------------------------------
limx, limy = 10, 10
psi = -45 * np.pi / 180
ab_ = 4

# ----------------------------------------------------------------------
# Scalar field generation
# ----------------------------------------------------------------------
n = 2
max_int = 20
dev = 10
mu = [40,40]

# Define the scalar field
S = -np.array([[1,0],[0,ab_]])
R = M_rot(psi)

sigma_func = sigma_gauss(mu=mu, max_intensity=max_int, dev=dev, S=S, R=R)
sigma_field = sigma(sigma_func)

# ----------------------------------------------------------------------
# Simulation parameters
# ----------------------------------------------------------------------
dt = 0.1
t0 = 0
t_sim_final = 60
n_agents = 150

# Generate different distributions
rc0 = [-35, -50]
border_noise = 0.6
lims = [limx, limy]

p1 = XY_distrib(n_agents, n, [0,0], lims, border_noise)
p1 = batman_distrib(n_agents, [0,0], lims)

# Initial state of the agents and number of agent
p0 = rc0 + p1
v0 = 4

In [ ]:
# ----------------------------------------------------------------------
# Numerical simulation
# ----------------------------------------------------------------------

# Initialize the simulation
sim = simulation_class1(sigma_field, n_agents, [t0, p0, v0], dt, True)

# Initialize the data collector
data_labels = ["pf", "rc", "e", "d", "sigma", "active", "l_sigma", 
               "rc_grad", "field_rot"]
data_col = data_collector(sim, data_labels)
data_col.collect()

# Execute numerical simulation
w1 = 0.2  # angular velocity of the scalar field (rad/s)
w2 = 0.1
t1, t2, t3, t4 = 3, 12, 15, 35
while (sim.tf <= t_sim_final - dt/10):
  if ((sim.tf >= t1) and (sim.tf < t2)):
    rot = w1*(sim.tf-t1)
    sim.sigma_field.rot = M_rot(rot)
  if ((sim.tf >= t3) and (sim.tf < t4)):
    sim.sigma_field.rot = M_rot(rot + w2*(sim.tf-t3))

  # Integrate new step
  sim.int_step()
  data_col.collect()

In [ ]:
# ----------------------------------------------------------------------
# Plot few steps of the simulation
# ----------------------------------------------------------------------
plot_class1(data_col, sim, t_list=[5,10,20,40], field_rot_sw=True)

In [ ]:
# ----------------------------------------------------------------------
# Animate the simulation
# ----------------------------------------------------------------------
anim = anim_class1(data_col, sim, anim_tf=40, res_label="HD", field_rot_sw=True)
HTML(anim.to_html5_video()) # It takes a looot of time...

### **SIM 4**: I'm batman!

In [ ]:
# ----------------------------------------------------------------------
# Maneuverability parameters
# ----------------------------------------------------------------------
mu = [40,40]
max_int = 20
mu = [40,40]
dev = [7,2]
n = 2

# ----------------------------------------------------------------------
# Scalar field generation
# ----------------------------------------------------------------------
psi = 100 * np.pi / 180
R2 = M_rot(psi)

sigma_func = sigma_fract(k=0.04, dev=[4,1], mu=mu)
sigma_field = sigma(sigma_func)
sigma_field.rot = R2

# ----------------------------------------------------------------------
# Simulation parameters
# ----------------------------------------------------------------------
dt = 0.1
t0 = 0
t_sim_final = 100
n_agents = 500
d_max = 14 #!!

# Generate different distributions
rc0 = [-35, -50]
r, h = 10, 2
border_noise = 0.6
lims = [15, 10]
d_max = 14 #!!

p_cir = circular_distrib(n_agents, n, [0,0], r, h, border_noise)
p_bat = batman_distrib(n_agents, [0,0], lims)

# Initial state of the agents and number of agent
p0 = rc0 + p_cir
v0 = 2.5

In [ ]:
# ----------------------------------------------------------------------
# Numerical simulation
# ----------------------------------------------------------------------

# Initialize the simulation
sim = simulation_class1(sigma_field, n_agents, [t0, p0, v0], dt, True, ang_noise=12, it_noise=20)

# Initialize the data collector
data_labels = ["pf", "rc", "e", "d", "sigma", "active", "l_sigma", 
               "rc_grad", "field_rot"]
data_col = data_collector(sim, data_labels)
data_col.collect()

# Execute numerical simulation
t1, t2 = 0, 20
sim.static_mod_shape = 1
while (sim.tf <= t_sim_final - dt/10):
  if (sim.tf >= t1) and (sim.tf <=t1+1):
    sim.Xd = p_bat
  if (sim.tf >= t2) and (sim.tf <=t2+1):
    sim.static_mod_shape = 0

  # Integrate new step
  sim.int_step()
  data_col.collect()

  # Check if any agent has to die!!
  ddata = data_col.get("d")[-1,:]
  for i in range(sim.N):
    if ddata[i] is not None:
      if ddata[i] > d_max:
        sim.active[i] = False
      else:
        random_num = np.random.random()
        if sim.tf >= t2+10 and random_num > 0.999:
          sim.active[i] = False

In [ ]:
# ----------------------------------------------------------------------
# Plot few steps of the simulation
# ----------------------------------------------------------------------
plot_class1(data_col, sim, t_list=[20,50,70,90], d_max=d_max)

In [ ]:
# ----------------------------------------------------------------------
# Animate the simulation
# ----------------------------------------------------------------------
anim = anim_class1(data_col, sim, anim_tf=80, res_label="2K", d_max=d_max, tail_frames=0)
HTML(anim.to_html5_video()) # It takes a looot of time...

## Simulation Class 2: Source-seeking with **unicycles** 

### **SIM 5**: Source-seeking with unicycles

In [ ]:
# ----------------------------------------------------------------------
# Scalar field generation
# ----------------------------------------------------------------------
n = 2
max_int = 20
mu = [40,40]
dev = 10

sigma_func = sigma_nonconvex(k=0.04, dev=dev, mu=mu)
sigma_field = sigma(sigma_func)

# ----------------------------------------------------------------------
# Simulation parameterself.frame_int_euler([p_dot_active])
# ----------------------------------------------------------------------
dt = 0.2
t0 = 0
t_sim_final = 100

# Initial states and number of agents
n_agents = 20
rc0 = [-35, -50]
r, h = 10, 2
lims = [15, 2]
border_noise = 0.6

p0_cir = circular_distrib(n_agents, n, [0,0], r, h, border_noise)
p0_sqr = XY_distrib(n_agents, n, [0,0], lims, border_noise)

p0 = rc0 + p0_sqr
v0 = 2
phi0 = np.random.rand(n_agents) * np.pi

kd = 0.2

In [ ]:
# ----------------------------------------------------------------------
# Numerical simulation
# ----------------------------------------------------------------------

# Initialize the simulation
sim = simulation_class2(sigma_field, n_agents, [t0, p0, v0, phi0], dt, kd)

# Initialize the data collector
data_labels = ["pf", "phif", "rc", "e", "d", "sigma", "omega", "l_sigma", "rc_grad"]
data_col = data_collector(sim, data_labels)
data_col.collect()

# Execute numerical simulation
while (sim.tf <= t_sim_final - dt/10):
  if (sim.tf >= 5) and (sim.tf <= 11):
    sim.Xd = p0_sqr

  # Integrate new step
  sim.int_step()
  data_col.collect()

In [ ]:
# ----------------------------------------------------------------------
# Plot few steps of the simulation
# ----------------------------------------------------------------------
plot_class2(data_col, sim, t_list=[0,20,40,90], alpha=0.5)

In [ ]:
# ----------------------------------------------------------------------
# Animate the simulation
# ----------------------------------------------------------------------
anim = anim_class2(data_col, sim, anim_tf=90, res_label="2K")
HTML(anim.to_html5_video()) # It takes a looot of time...